# 10 — Manual homography and its validation (Phase 3)

The plan has Ro click four pitch landmarks per frame. Ro waived the clicking (2026-09-27), so the correspondences come from the **human keypoint annotations** of the pitch dataset, which were made by annotators clicking the same landmarks. That turns 3 frames into all **50 test-split frames** (matches 121364 and 744b27, which no model trained on).

Per frame, on the penalty area the camera shows best:
- **Fit on 4:** the penalty-box corners (goal line × box sides, box front corners) — geometry fixed by the laws of the game. Plain least squares, as for hand clicks.
- **Held-out reprojection error** on the other labelled penalty-area points (goal-area corners, penalty spot, penalty-arc intersections), in pixels and metres. Points that depend on the assumed 105 × 68 m pitch (corners, halfway line, centre circle) are reported separately, so a pitch-size mismatch is not mistaken for calibration error.
- **Reconstructed dimension check:** the goal-area width (officially 18.32 m) measured through the homography, error in cm.
- **Sensitivity:** every fit point moved by 1–5 px in random directions (300 trials), refit, and the shift of an offside line at the far touchline measured in cm — for a line at the penalty-box front and one 30 m from goal.

In [1]:
FRAMES = "frames/pitch_test"
OUT = "results/homography_manual"
PX_LEVELS = [1, 2, 3, 4, 5]
TRIALS = 300

In [2]:
import json, pathlib
import numpy as np, pandas as pd, cv2
import ballhawk_pitch as bp

out = pathlib.Path(OUT); out.mkdir(parents=True, exist_ok=True)
SIDES = {  # fit: penalty-box corners; held: other law-fixed penalty-area points; line_x: box front, 30 m from goal
    "left":  {"fit": [1, 4, 12, 9],    "held": [2, 3, 6, 7, 8, 10, 11],        "ga": (2, 3),   "lines": [bp.PENALTY_L, 30.0]},
    "right": {"fit": [25, 28, 20, 17], "held": [26, 27, 22, 23, 21, 18, 19],   "ga": (26, 27), "lines": [bp.LENGTH - bp.PENALTY_L, bp.LENGTH - 30.0]},
}
SIZE_DEP = [0, 5, 24, 29, 13, 14, 15, 16, 30, 31]  # depend on the assumed 105 x 68 m pitch

def labelled(img):
    h, w = cv2.imread(str(img)).shape[:2]
    k = np.array(open(pathlib.Path(FRAMES) / "labels" / f"{img.stem}.txt").read().split(), float)[5:].reshape(-1, 3)
    vis = set(np.flatnonzero(k[:, 2] > 0).tolist())
    return vis, k[:, :2] * [w, h]

In [3]:
rows, sens_rows = [], []
for img in sorted(pathlib.Path(FRAMES, "images").iterdir()):
    vis, xy = labelled(img)
    side = max(SIDES, key=lambda s: len(vis & set(SIDES[s]["fit"] + SIDES[s]["held"])))
    S = SIDES[side]
    if not set(S["fit"]) <= vis:
        rows.append({"image": img.name, "side": side, "status": "fewer than 4 box corners labelled"}); continue
    H, _ = bp.fit_homography(xy[S["fit"]], bp.PITCH_XY[S["fit"]])
    held = [i for i in S["held"] if i in vis]
    dep = [i for i in SIZE_DEP if i in vis]
    m_h, px_h = bp.reprojection_error(H, xy[held], bp.PITCH_XY[held]) if held else ([], [])
    m_d, _ = bp.reprojection_error(H, xy[dep], bp.PITCH_XY[dep]) if dep else ([], [])
    ga = None
    if set(S["ga"]) <= vis:
        a, b = bp.project(H, xy[list(S["ga"])])
        ga = (np.linalg.norm(a - b) - bp.GOAL_AREA_W) * 100
    rows.append({"image": img.name, "side": side, "status": "ok", "held_out_points": len(held),
                 "held_median_m": float(np.median(m_h)) if held else np.nan, "held_median_px": float(np.median(px_h)) if held else np.nan,
                 "size_dependent_median_m": float(np.median(m_d)) if dep else np.nan, "goal_area_width_error_cm": ga})
    for line_x in S["lines"]:
        for px, (mean_cm, p95_cm) in bp.line_shift_sensitivity(xy[S["fit"]], bp.PITCH_XY[S["fit"]], line_x, far_y=0.0,
                                                               px_levels=PX_LEVELS, trials=TRIALS).items():
            sens_rows.append({"image": img.name, "line_from_goal_m": round(min(line_x, bp.LENGTH - line_x), 1),
                              "click_error_px": px, "mean_shift_cm": mean_cm, "p95_shift_cm": p95_cm})
df, sens = pd.DataFrame(rows), pd.DataFrame(sens_rows)
df.to_csv(out / "frames.csv", index=False); sens.to_csv(out / "sensitivity.csv", index=False)
ok = df[df.status == "ok"]
print(df.status.value_counts().to_dict()); ok.describe().round(2)

{'fewer than 4 box corners labelled': 30, 'ok': 20}


,held_out_points,held_median_m,held_median_px,size_dependent_median_m,goal_area_width_error_cm
count,20.0,20.00,20.00,20.00,20.00
mean,7.0,0.42,3.41,0.86,-7.42
std,0.0,0.17,1.06,0.44,20.41
min,7.0,0.14,1.68,0.21,-43.58
25%,7.0,0.33,2.71,0.56,-22.57
50%,7.0,0.39,3.67,0.82,-8.30
75%,7.0,0.51,4.12,1.06,3.69
max,7.0,0.78,5.11,1.78,36.31


In [4]:
curve = sens.groupby(["line_from_goal_m", "click_error_px"]).agg(median_mean_cm=("mean_shift_cm", "median"),
                                                               median_p95_cm=("p95_shift_cm", "median")).round(1)
per_px = {f"{d:g} m from goal": float(np.median(sens[sens.line_from_goal_m == d].pipe(lambda s: s.mean_shift_cm / s.click_error_px)))
          for d in sorted(sens.line_from_goal_m.unique())}
summary = {"frames": int(len(df)), "evaluated": int(len(ok)), "source": "human keypoint annotations (pitch test split) in place of hand clicks",
           "held_out_median_m": float(ok.held_median_m.median()), "held_out_p90_m": float(ok.held_median_m.quantile(0.9)),
           "held_out_median_px": float(ok.held_median_px.median()),
           "size_dependent_median_m": float(ok.size_dependent_median_m.median()),
           "goal_area_width_median_abs_error_cm": float(ok.goal_area_width_error_cm.abs().median()),
           "goal_area_width_frames": int(ok.goal_area_width_error_cm.notna().sum()),
           "line_shift_cm_per_px_median": per_px,
           "curve": {f"{d:g}m_{p}px": {"mean_cm": r.median_mean_cm, "p95_cm": r.median_p95_cm} for (d, p), r in curve.iterrows()}}
(out / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps({k: v for k, v in summary.items() if k != "curve"}, indent=2)); curve

{
  "frames": 50,
  "evaluated": 20,
  "source": "human keypoint annotations (pitch test split) in place of hand clicks",
  "held_out_median_m": 0.39204928531966654,
  "held_out_p90_m": 0.6904965788843649,
  "held_out_median_px": 3.6698854584744685,
  "size_dependent_median_m": 0.8183065837802421,
  "goal_area_width_median_abs_error_cm": 13.444455529503507,
  "goal_area_width_frames": 20,
  "line_shift_cm_per_px_median": {
    "16.5 m from goal": 9.809137461256544,
    "30 m from goal": 11.76091619483546
  }
}


median_mean_cm  median_p95_cm
line_from_goal_m click_error_px                               
16.5             1                          9.7           17.0
                 2                         19.5           33.0
                 3                         28.2           50.9
                 4                         40.3           66.8
                 5                         49.4           83.4
30.0             1                         11.6           23.5
                 2                         23.7           49.2
                 3                         34.4           73.6
                 4                         46.4           95.7
                 5                         61.5          119.9